# Where Philly Smells Like Coffee

Day 19 · Smell. Every coffee spot in Philadelphia mapped in OpenStreetMap, drawn as a cloud of coffee smell drifting
out over the streets. Run `python scripts\fetch_week3.py day19` first.

## 1 · Settings

In [ ]:
DAY      = 19
PROMPT   = "Smell"
SLUG     = "smell-philly-coffee"
NOTEBOOK = "day19_smell_philly_coffee.ipynb"
TITLE    = "Where Philly\nSmells Like Coffee"
CREDITS  = "Cafés, streets and water: © OpenStreetMap contributors (ODbL). City boundary: US Census Bureau."

SNIFF_M  = 250          # "you can smell it": within this many metres of a coffee spot (about a 3-minute walk)
CLOUD_M  = 220          # how far each café's smell spreads on the map (Gaussian blur, metres)
NOT_COFFEE = ("tea", "boba", "chicha", "gong cha", "mr. wish", "matcha", "bambu", "mango mango", "kung fu")
KEEP     = ("ray’s cafe", "ray's cafe")   # coffee spots with "tea" in the name
CHAINS   = ("Starbucks",)        # drawn apart from everyone else
PLACES   = {"Center City": (39.9515, -75.1650)}

PAGE, INK, SOFT = "#f4ece1", "#2a1a12", "#8a7464"
STREET, WATER_C = "#d9c9b6", "#dfe6e4"
CLOUD = ["#f4ece1", "#ead2b4", "#d6a878", "#a8693a", "#5b3420"]   # faint to strong
CHAIN_C = "#5b6b5f"

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from mapkit import poster, publish
from mapkit.paths import PROCESSED

M = "EPSG:32618"                       # metres, UTM zone 18N
cafes = gpd.read_file(PROCESSED / "philly_coffee.geojson").to_crs(M)
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'philly'").to_crs(M)
streets = gpd.read_file(PROCESSED / "philly_streets.geojson").to_crs(M)
water = gpd.read_file(PROCESSED / "philly_water.geojson").to_crs(M)
print(f"{len(cafes)} places tagged café or coffee shop · {len(streets):,} street pieces")

## 3 · Analysis, every number on the map comes from here

In [ ]:
label = (cafes["name"].fillna("") + " " + cafes["brand"].fillna("") + " " + cafes["cuisine"].fillna("")).str.lower()
tea = label.str.contains("|".join(NOT_COFFEE)) & ~label.str.contains("coffee") & ~label.str.contains("|".join(KEEP))
print("Left out as tea or dessert, not coffee:", sorted(cafes.loc[tea, "name"].fillna("(no name)").unique()))
coffee = cafes[~tea].copy()
coffee = coffee[coffee.within(city.union_all().buffer(50))]
coffee["chain"] = coffee["brand"].fillna(coffee["name"]).isin(CHAINS)
N, N_CHAIN = len(coffee), int(coffee.chain.sum())

# share of street length you can smell coffee on
sniff = coffee.buffer(SNIFF_M).union_all()
streets["m"] = streets.length
in_city = streets[streets.intersects(city.union_all())]
SMELL_SHARE = in_city.intersection(sniff).length.sum() / in_city.m.sum() * 100
SMELL_SHARE_INDIE = in_city.intersection(coffee[~coffee.chain].buffer(SNIFF_M).union_all()).length.sum() / in_city.m.sum() * 100

# busiest corner: most coffee spots within SNIFF_M of one café
xy = np.c_[coffee.geometry.x, coffee.geometry.y]
from scipy.spatial import cKDTree
counts = np.array([len(i) for i in cKDTree(xy).query_ball_point(xy, SNIFF_M)])
busiest = coffee.iloc[counts.argmax()]

print(f"{N} coffee spots: {N_CHAIN} Starbucks, {N - N_CHAIN} everyone else ({(N - N_CHAIN) / N_CHAIN:.1f} for every Starbucks)")
print(f"You can smell coffee (within {SNIFF_M} m) on {SMELL_SHARE:.0f}% of Philly's street length; "
      f"{SMELL_SHARE_INDIE:.0f}% if only the non-Starbucks count")
print(f"Busiest corner: {counts.max()} coffee spots within {SNIFF_M} m of {busiest['name']}")

## 4 · Map

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.ABRIL, poster.POPPINS, poster.POPPINS_BOLD
fig, ax = poster.page(PAGE)
ax.set_position([0.0, 0.07, 1.0, 0.76])

# the smell: every coffee spot blurred into a cloud
x0, y0, x1, y1 = city.total_bounds
res = 25
gx = np.arange(x0, x1 + res, res); gy = np.arange(y0, y1 + res, res)
H, _, _ = np.histogram2d(coffee.geometry.y, coffee.geometry.x, bins=[gy, gx])
cloud = gaussian_filter(H, CLOUD_M / res)
cloud = cloud / cloud.max()
cmap = LinearSegmentedColormap.from_list("coffee", CLOUD)
im = ax.imshow(np.sqrt(cloud), origin="lower", extent=(x0, x1, y0, y1), cmap=cmap, vmin=0.04, vmax=1, zorder=1,
               interpolation="bilinear")
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MPath
geom = city.union_all()
polys = getattr(geom, "geoms", [geom])
im.set_clip_path(PathPatch(MPath.make_compound_path(*[MPath(np.asarray(p.exterior.coords)) for p in polys]),
                           transform=ax.transData, fc="none", ec="none"))
water.plot(ax=ax, color=WATER_C, lw=0, zorder=2)
in_city.plot(ax=ax, color=INK, lw=0.18, alpha=0.22, zorder=3)
city.boundary.plot(ax=ax, color=INK, lw=0.8, zorder=4)
indie = coffee[~coffee.chain]; chain = coffee[coffee.chain]
ax.scatter(indie.geometry.x, indie.geometry.y, s=9, color=CLOUD[-1], edgecolor=PAGE, lw=0.4, zorder=5)
ax.scatter(chain.geometry.x, chain.geometry.y, s=16, facecolor=PAGE, edgecolor=CHAIN_C, lw=1.1, zorder=6)
ax.set_xlim(x0 - 500, x1 + 500); ax.set_ylim(y0 - 500, y1 + 500); ax.set_aspect("equal"); ax.set_axis_off()

fig.text(0.06, 0.955, TITLE, color=INK, fontproperties=F(TITLE_F, 52), va="top", linespacing=0.82)
fig.text(0.06, 0.835, f"{N - N_CHAIN} coffee spots that aren't Starbucks. {N_CHAIN} that are.", color=SOFT,
         fontproperties=F(TEXT_F, 15), va="top")
for name, (la, lo) in PLACES.items():
    p = gpd.GeoSeries(gpd.points_from_xy([lo], [la]), crs=4326).to_crs(M).iloc[0]
    poster.halo_text(ax, p.x + 500, p.y + 700, name, color=INK, path=TEXT_B, size=11, halo=PAGE, halo_width=3,
                     ha="left", va="bottom", zorder=8)

# legend, lower right
lx, ly = 0.66, 0.20
ax.figure.add_artist(plt.matplotlib.patches.Circle((lx, ly), 0.005, transform=fig.transFigure, color=CLOUD[-1]))
fig.text(lx + 0.014, ly, "Coffee spot", color=INK, fontproperties=F(TEXT_F, 12), va="center")
ax.figure.add_artist(plt.matplotlib.patches.Circle((lx, ly - 0.025), 0.0055, transform=fig.transFigure, facecolor=PAGE,
                                                   edgecolor=CHAIN_C, lw=1.1))
fig.text(lx + 0.014, ly - 0.025, "Starbucks", color=INK, fontproperties=F(TEXT_F, 12), va="center")
fig.add_artist(plt.Rectangle((lx - 0.005, ly - 0.068), 0.16, 0.014, transform=fig.transFigure,
                             color="none"))
for i, c in enumerate(CLOUD[1:]):
    fig.add_artist(plt.Rectangle((lx - 0.005 + i * 0.04, ly - 0.07), 0.04, 0.014, transform=fig.transFigure, color=c, lw=0))
fig.text(lx - 0.005, ly - 0.078, "faint", color=SOFT, fontproperties=F(TEXT_F, 10), va="top")
fig.text(lx + 0.155, ly - 0.078, "strong", color=SOFT, fontproperties=F(TEXT_F, 10), va="top", ha="right")
fig.text(lx - 0.005, ly - 0.045, "COFFEE IN THE AIR", color=SOFT, fontproperties=F(TEXT_B, 9.5), va="center")

# credit, set by hand because Poppins sits taller than the helper expects
fig.text(0.06, 0.020, "Bubble tea and dessert shops left out. The smell is a drawing, not a measurement.",
         color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.038, "Cafés, streets, water © OpenStreetMap contributors · city boundary US Census",
         color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.058, "Map by Gautami Kasat", color=INK, fontproperties=F(TEXT_B, 12), va="bottom")
plt.show()

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{N - N_CHAIN} coffee spots that aren't Starbucks, {N_CHAIN} that are; coffee within {SNIFF_M} m on {SMELL_SHARE:.0f}% of streets",
             series="city", tool="Python", notebook=NOTEBOOK, credits=CREDITS)